# 02 - Exploratory Data Analysis & Kinematic Modeling

**Project**: Intelligent Driver Activity Monitoring & Safety Agent  
**Objective**: In-depth kinematic analysis, orientation-invariant feature relationships, outlier characterization, and leakage-safe evaluation protocols.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
possible_dirs = [
    Path("../dataset/raw"),
    Path("../../dataset/raw"),
    Path("driver-monitoring-agent/dataset/raw"),
    Path("dataset/raw"),
    Path(r"D:/driver ml model/driver-monitoring-agent/dataset/raw"),
]
base_dir = next((p for p in possible_dirs if p.exists()), None)
print("Dataset loaded from:", base_dir.resolve())


## 1. Load Multi-Sensor Continuous Telemetry
Load Accelerometer and Gyroscope streams for Driver-1 (100 Hz continuous logging).


In [ ]:
acc_df = pd.read_csv(base_dir / "drivers" / "Driver-1" / "Accelerometer.csv")
gyr_df = pd.read_csv(base_dir / "drivers" / "Driver-1" / "Gyroscope.csv")
print(f"Driver-1 Accel shape: {acc_df.shape}, Gyro shape: {gyr_df.shape}")


## 2. Rotation-Invariant Kinematic Feature Extraction
Because phone orientation inside the utility box is uncalibrated, Euclidean vector magnitudes provide orientation-invariant kinetic signals:
$$\|a\| = \sqrt{X^2 + Y^2 + Z^2}, \quad \|\omega\| = \sqrt{\omega_X^2 + \omega_Y^2 + \omega_Z^2}$$


In [ ]:
acc_df["Norm"] = np.sqrt(acc_df["X"]**2 + acc_df["Y"]**2 + acc_df["Z"]**2)
gyr_df["Norm"] = np.sqrt(gyr_df["X"]**2 + gyr_df["Y"]**2 + gyr_df["Z"]**2)

print("Accelerometer Summary Statistics (m/s^2):")
print(acc_df[["X", "Y", "Z", "Norm"]].describe())
print("\nGyroscope Summary Statistics (rad/s):")
print(gyr_df[["X", "Y", "Z", "Norm"]].describe())


## 3. Continuous Telemetry Waveform Analysis
Visualizing 30 seconds of high-rate telemetry (3,000 samples at 100 Hz).


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
time_sec = np.arange(3000) / 100.0  # 100 Hz

axes[0].plot(time_sec, acc_df["X"].iloc[:3000], label="Acc X", color="#1976d2", alpha=0.8)
axes[0].plot(time_sec, acc_df["Y"].iloc[:3000], label="Acc Y", color="#388e3c", alpha=0.8)
axes[0].plot(time_sec, acc_df["Z"].iloc[:3000], label="Acc Z (Gravity axis)", color="#f57c00", alpha=0.8)
axes[0].set_ylabel("Acceleration (m/s²)")
axes[0].set_title("Driver-1 Telemetry: 30-Second Acceleration Traces (100 Hz sampling)", fontweight="bold")
axes[0].legend(loc="upper right")

axes[1].plot(time_sec, gyr_df["X"].iloc[:3000], label="Gyro X", color="#7b1fa2", alpha=0.8)
axes[1].plot(time_sec, gyr_df["Y"].iloc[:3000], label="Gyro Y", color="#c2185b", alpha=0.8)
axes[1].plot(time_sec, gyr_df["Z"].iloc[:3000], label="Gyro Z", color="#d32f2f", alpha=0.8)
axes[1].set_ylabel("Angular Velocity (rad/s)")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("Driver-1 Telemetry: 30-Second Angular Velocity Traces (100 Hz sampling)", fontweight="bold")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()


## 4. Cross-Axis Correlation Matrix


In [ ]:
n = min(len(acc_df), len(gyr_df))
combined = pd.DataFrame({
    "Acc_X": acc_df["X"].iloc[:n],
    "Acc_Y": acc_df["Y"].iloc[:n],
    "Acc_Z": acc_df["Z"].iloc[:n],
    "Acc_Norm": acc_df["Norm"].iloc[:n],
    "Gyro_X": gyr_df["X"].iloc[:n],
    "Gyro_Y": gyr_df["Y"].iloc[:n],
    "Gyro_Z": gyr_df["Z"].iloc[:n],
    "Gyro_Norm": gyr_df["Norm"].iloc[:n],
})
plt.figure(figsize=(8, 6))
sns.heatmap(combined.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Sensor Channel Correlation Matrix", fontweight="bold")
plt.show()


## 5. Inter-Driver Variance Comparison (Table 3 Subjects)


In [ ]:
comp = []
for i in range(1, 8):
    p_a = base_dir / "drivers" / f"Driver-{i}" / "Accelerometer.csv"
    p_g = base_dir / "drivers" / f"Driver-{i}" / "Gyroscope.csv"
    if p_a.exists() and p_g.exists():
        a = pd.read_csv(p_a)
        g = pd.read_csv(p_g)
        a_norm = np.sqrt(a["X"]**2 + a["Y"]**2 + a["Z"]**2)
        g_norm = np.sqrt(g["X"]**2 + g["Y"]**2 + g["Z"]**2)
        comp.append({
            "Driver": f"Driver-{i}",
            "Accel_Std": round(float(a_norm.std()), 3),
            "Accel_Max": round(float(a_norm.max()), 2),
            "Gyro_Std": round(float(g_norm.std()), 4),
            "Gyro_Max": round(float(g_norm.max()), 2),
        })
df_comp = pd.DataFrame(comp)
print(df_comp)


## 6. Leakage-Safe Problem Formulation & Validation Strategy
1. **Activity Labels**: No pre-assigned ground-truth activity labels exist. Normal/Aggressive/Risky must be framed as unsupervised kinematic anomaly detection or event clustering.
2. **Subject Splitting**: `drivers/` contains 7 distinct drivers; `daywise/` contains a single driver over 7 days.
3. **Recommended Evaluation**: Multi-subject Leave-One-Driver-Out CV on `drivers/` or train on Days 1-5 and test on Days 6-7 on `daywise/`.
